In [ ]:
import pandas as pd
from pathlib import Path
from typing import List
import regex as re
import math
import csv

# Helper Code

In [ ]:
PATTERN = re.compile("([SNsn])(\\d{1,3})° (\\d{1,2})\\' (\\d{1,2}\\.\\d*)\"([EWwe])")

def load_csv(file: Path) -> pd.DataFrame:
  return pd.read_csv(file)

def load_excel(file: Path) -> pd.DataFrame:
  return pd.read_excel(file)

"""Load all data files in current working directory into dataframes"""
def load_all() -> List[pd.DataFrame]:
  cwd = Path.cwd()
  dfs = []
  files = []

  for datafile in cwd.iterdir():
    try:
      if datafile.suffix == '.xlsx' or datafile.suffix == '.xls':
        dfs.append(load_excel(datafile))
        files.append(datafile)
      elif datafile.suffix == '.csv':
        dfs.append(load_csv(datafile))
        files.append(datafile)
      else:
        print("WARNING: Unknown file extension. Use .csv, .xls, or .xlsx. Ignoring...")
    except Exception as e:
      print(f"Exception: {e}")
      print(f"Cannot load file: {datafile}, it maybe empty...")

  return dfs, files

def round(m) -> str:
  c1, dd, mm, ss, c2 = m.groups()
  fl_seconds = float(ss)
  i_minutes = int(mm)
  i_degrees = int(dd)
  frac_seconds, i_seconds = math.modf(fl_seconds)
  if frac_seconds == 0.5:
    i_seconds += int(i_seconds % 2 != 0)
  elif frac_seconds > 0.5:
    i_seconds += 1

  i_seconds = int(i_seconds)

  if i_seconds == 60:
    i_seconds = 0
    i_minutes += 1

  if i_minutes == 60:
    i_minutes = 0
    i_degrees += 1

  return f"{c1}{i_degrees:02}°{i_minutes:02}'{i_seconds:02}\"{c2}"

def find_and_replace_dec_seconds(cell) -> str:
  if not isinstance(cell, str):
    return cell

  new = PATTERN.sub(round, cell)
  return PATTERN.sub(round, cell)

def write_updates(dfs: List[pd.DataFrame], files: List[str]):
  for df, filename in zip(dfs, files):
    if filename.suffix == ".csv":
      df.to_csv(filename, quoting=csv.QUOTE_NONE, index=False)
    else:
      df.to_excel(filename, index=False)

# Load and convert

In [ ]:
datafiles, filenames = load_all()

for i in range(len(datafiles)):
  datafiles[i] = datafiles[i].map(find_and_replace_dec_seconds)

write_updates(datafiles, filenames)